In [4]:
#1. Import Libraries
import pandas as pd
import numpy as np

# For visualization (optional at this stage)
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
#2. first install the kagglehub package
!pip install kagglehub
import kagglehub

# Download latest version
path = kagglehub.dataset_download("sobhanmoosavi/us-accidents")

print("Path to dataset files:", path)

Defaulting to user installation because normal site-packages is not writeable


100%|██████████| 653M/653M [03:25<00:00, 3.34MB/s] 

Extracting files...


Path to dataset files: C:\Users\firao\.cache\kagglehub\datasets\sobhanmoosavi\us-accidents\versions\13


In [11]:
import os

print(os.listdir(path))

['US_Accidents_March23.csv']


In [12]:
# 3. Load Data
file_path = os.path.join(path, "US_Accidents_March23.csv")

In [13]:
# Load a sample first (100k rows) to test
df = pd.read_csv(file_path, nrows=100000, low_memory=False)
print("Sample shape:", df.shape)
df.head()

Sample shape: (100000, 46)


,ID,Source,Severity,Start_Time,End_Time,Start_Lat,Start_Lng,End_Lat,End_Lng,Distance(mi),...,Roundabout,Station,Stop,Traffic_Calming,Traffic_Signal,Turning_Loop,Sunrise_Sunset,Civil_Twilight,Nautical_Twilight,Astronomical_Twilight
0,A-1,Source2,3,2016-02-08 05:46:00,2016-02-08 11:00:00,39.865147,-84.058723,NaN,NaN,0.01,...,False,False,False,False,False,False,Night,Night,Night,Night
1,A-2,Source2,2,2016-02-08 06:07:59,2016-02-08 06:37:59,39.928059,-82.831184,NaN,NaN,0.01,...,False,False,False,False,False,False,Night,Night,Night,Day
2,A-3,Source2,2,2016-02-08 06:49:27,2016-02-08 07:19:27,39.063148,-84.032608,NaN,NaN,0.01,...,False,False,False,False,True,False,Night,Night,Day,Day
3,A-4,Source2,3,2016-02-08 07:23:34,2016-02-08 07:53:34,39.747753,-84.205582,NaN,NaN,0.01,...,False,False,False,False,False,False,Night,Day,Day,Day
4,A-5,Source2,2,2016-02-08 07:39:07,2016-02-08 08:09:07,39.627781,-84.188354,NaN,NaN,0.01,...,False,False,False,False,True,False,Day,Day,Day,Day


In [14]:
# 4. Select Relevant Columns
cols_to_keep = [
    "Severity", "Start_Time", "End_Time", "Start_Lat", "Start_Lng",
    "City", "County", "State", "Distance(mi)",
    "Weather_Condition", "Temperature(F)", "Humidity(%)",
    "Visibility(mi)", "Wind_Speed(mph)", "Pressure(in)",
    "Sunrise_Sunset"
]

df = df[cols_to_keep].copy()
print("Shape after keeping useful columns:", df.shape)

Shape after keeping useful columns: (100000, 16)


In [24]:
#Check missing values
df.isnull().sum()

Severity             0
Start_Time           0
End_Time             0
Start_Lat            0
Start_Lng            0
City                 0
County               0
State                0
Distance(mi)         0
Weather_Condition    0
Temperature(F)       0
Humidity(%)          0
Visibility(mi)       0
Wind_Speed(mph)      0
Pressure(in)         0
Sunrise_Sunset       0
Year                 0
Month                0
DayOfWeek            0
Hour                 0
DurationHrs          0
Is_Day               0
Weather_Simple       0
dtype: int64

In [27]:
# 5. Handle Missing Data

# Drop columns with >40% missing
threshold = 0.4
df = df.loc[:, df.isnull().mean() < threshold]

# Fill numerical NaNs with median
num_cols = df.select_dtypes(include=np.number).columns
for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

# Fill categorical NaNs with mode
cat_cols = df.select_dtypes(include="object").columns
for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

print("Missing values handled.")

Missing values handled.


In [18]:
#6. Feature Engineering
# Convert Start_Time to datetime
df["Start_Time"] = pd.to_datetime(df["Start_Time"], errors="coerce")

In [19]:
# Extract time features
df["Year"] = df["Start_Time"].dt.year
df["Month"] = df["Start_Time"].dt.month
df["DayOfWeek"] = df["Start_Time"].dt.dayofweek
df["Hour"] = df["Start_Time"].dt.hour

In [20]:
# Accident duration (hours)
df["End_Time"] = pd.to_datetime(df["End_Time"], errors="coerce")
df["DurationHrs"] = (df["End_Time"] - df["Start_Time"]).dt.total_seconds() / 3600
df["DurationHrs"] = df["DurationHrs"].clip(upper=72)  # cap extreme outliers at 3 days

# Encode day/night
df["Is_Day"] = df["Sunrise_Sunset"].apply(lambda x: 1 if x == "Day" else 0)


In [21]:
# Simplify weather categories
def simplify_weather(condition):
    if pd.isna(condition):
        return "Unknown"
    c = condition.lower()
    if "rain" in c or "storm" in c:
        return "Rain"
    elif "snow" in c or "sleet" in c or "ice" in c:
        return "Snow"
    elif "fog" in c or "haze" in c:
        return "Fog"
    elif "clear" in c or "fair" in c:
        return "Clear"
    elif "cloud" in c or "overcast" in c:
        return "Cloudy"
    else:
        return "Other"

df["Weather_Simple"] = df["Weather_Condition"].apply(simplify_weather)

print("Feature engineering complete.")
df.head()

Feature engineering complete.


,Severity,Start_Time,End_Time,Start_Lat,Start_Lng,City,County,State,Distance(mi),Weather_Condition,...,Wind_Speed(mph),Pressure(in),Sunrise_Sunset,Year,Month,DayOfWeek,Hour,DurationHrs,Is_Day,Weather_Simple
0,3,2016-02-08 05:46:00,2016-02-08 11:00:00,39.865147,-84.058723,Dayton,Montgomery,OH,0.01,Light Rain,...,6.9,29.68,Night,2016,2,0,5,5.233333,0,Rain
1,2,2016-02-08 06:07:59,2016-02-08 06:37:59,39.928059,-82.831184,Reynoldsburg,Franklin,OH,0.01,Light Rain,...,6.9,29.65,Night,2016,2,0,6,0.500000,0,Rain
2,2,2016-02-08 06:49:27,2016-02-08 07:19:27,39.063148,-84.032608,Williamsburg,Clermont,OH,0.01,Overcast,...,3.5,29.67,Night,2016,2,0,6,0.500000,0,Cloudy
3,3,2016-02-08 07:23:34,2016-02-08 07:53:34,39.747753,-84.205582,Dayton,Montgomery,OH,0.01,Mostly Cloudy,...,4.6,29.64,Night,2016,2,0,7,0.500000,0,Cloudy
4,2,2016-02-08 07:39:07,2016-02-08 08:09:07,39.627781,-84.188354,Dayton,Montgomery,OH,0.01,Mostly Cloudy,...,3.5,29.65,Day,2016,2,0,7,0.500000,1,Cloudy


In [35]:
# 7. Save Cleaned Dataset

# Save smaller, faster format (Parquet)
df.to_parquet("us_accidents_clean.parquet", index=False)

# Optional: also save CSV
df.to_csv("us_accidents_clean.csv", index=False)

print("Clean dataset saved!")

Clean dataset saved!


In [23]:
# 8. Reload Later
df_clean = pd.read_parquet("us_accidents_clean.parquet")
print("Reloaded shape:", df_clean.shape)
df_clean.head()

Reloaded shape: (100000, 23)


,Severity,Start_Time,End_Time,Start_Lat,Start_Lng,City,County,State,Distance(mi),Weather_Condition,...,Wind_Speed(mph),Pressure(in),Sunrise_Sunset,Year,Month,DayOfWeek,Hour,DurationHrs,Is_Day,Weather_Simple
0,3,2016-02-08 05:46:00,2016-02-08 11:00:00,39.865147,-84.058723,Dayton,Montgomery,OH,0.01,Light Rain,...,6.9,29.68,Night,2016,2,0,5,5.233333,0,Rain
1,2,2016-02-08 06:07:59,2016-02-08 06:37:59,39.928059,-82.831184,Reynoldsburg,Franklin,OH,0.01,Light Rain,...,6.9,29.65,Night,2016,2,0,6,0.500000,0,Rain
2,2,2016-02-08 06:49:27,2016-02-08 07:19:27,39.063148,-84.032608,Williamsburg,Clermont,OH,0.01,Overcast,...,3.5,29.67,Night,2016,2,0,6,0.500000,0,Cloudy
3,3,2016-02-08 07:23:34,2016-02-08 07:53:34,39.747753,-84.205582,Dayton,Montgomery,OH,0.01,Mostly Cloudy,...,4.6,29.64,Night,2016,2,0,7,0.500000,0,Cloudy
4,2,2016-02-08 07:39:07,2016-02-08 08:09:07,39.627781,-84.188354,Dayton,Montgomery,OH,0.01,Mostly Cloudy,...,3.5,29.65,Day,2016,2,0,7,0.500000,1,Cloudy
